# Battle 采样节点（rollout / eval）— Kaggle / Colab（薄 cell 版）

这台云机**只采样本与跑评估**，不训练：它起 `tools/agent/sampler-agent.ts`，把它的 HTTP 端口暴露到
公网（Colab 优先 tailscale；其余平台、或 tailscale 引导失败/入站不可达 ⇒ cloudflared 隧道），再
**自动**把 `{id,url,authKey}` 注册进训练机的 `nn-training/rl-config.json`
（`POST /admin/nodes/register`，hub 侧 upsert `nodes[]`）。

| 步骤 | 谁做 | 怎么做 |
|---|---|---|
| 下载代码库 | 本 cell | `git clone --depth 1`（**全仓**；理由见 cell 内注释），或 `code_zip` 离线归档 |
| 装 bun / cloudflared | `remote/rollout_node.py` | 公网下载一律包在 `platform_net_env()` 里（代理注入后公网不通） |
| 网络通道 | `remote/rollout_node.py` | tailscale 优先 → 注册被 hub ping 门拒（422）⇒ 回落 cloudflared 并重注册 |
| 并发度 | `remote/rollout_node.py` | `platform_utils.cpu_worker_slots()`（rollout/eval 的**唯一口径**） |
| 保活 / 日志 | 本 cell | 心跳打印 + 同时落文件（Colab 输出会连着丢） |
| 自动注册 | `remote/rollout_node.py` | `HUB_IP`/`HUB_TOKEN` secret 齐 ⇒ 自动；缺 ⇒ 打印 ★URL + ★authKey 走手工路径 |

**逻辑住在仓库**：`nn-training/remote/rollout_node.py` —— cell 只留 CFG / 凭据 / 保活 / clone / 调用。
改逻辑请改那个文件，**别在这里内联**（`tests/remote/test_tailscale_boot_bun.py` 会红）。

**用法**：填 `CFG.repo`（+ 私有仓的 `github_token`）→ 配好 secret `HUB_IP` / `HUB_TOKEN`
（tailscale 通道另需 `TS_AUTHKEY`）→ Run All → 等日志出现 `★ 已自动注册`。

## GitHub 直连不通（阿里云 DSW / 国内云机）

`github.com` 超时（TLS 中断 / 443 不通）时**不用填任何开关**，默认自动换路：

1. **clone**：候选（直连 + 内置镜像）**并行** `git ls-remote` 探活（25s 封顶），按耗时取最快；
   clone 失败再顺延下一候选。
2. **bun**：候选（直连 + 内置镜像）先并行拉 256KB 小样测吞吐，按快慢依次带 `GITHUB=<基址>`
   重跑官方 install.sh；仍失败 ⇒ 自动回落 npmmirror（`registry.npmmirror.com`）。
3. **cloudflared**：同上并行测速（256KB），按实测快慢依次下载，失败顺延；每跳连接超时 8s。

需要覆盖时才有这几个口子（都留空即可）：`CFG.github_base` 换自建/其它前缀式镜像；
`CFG.bun_mirror` 换 bun 二进制镜像目录；`CFG.net_proxy` 给出站代理；`CFG.code_zip` 完全离线
（私有仓 / 内网：本机把仓库根打成 zip（**不要顶层目录**）上传，跳过全部 git 操作）。
镜像服务不转发私有仓认证 ⇒ 私有仓请走 `code_zip` 或直连 + `github_token`。

**手工兜底**（没配 hub 时照旧可用）：把日志里的 `★ URL` 与 `★ authKey` 写进本机
`nn-training/rl-config.json`：

```json
{ "nodes": [ { "id": "rollout-colab", "url": "<URL>", "authKey": "<authKey>" } ] }
```

（`concurrency` 不填 = 按节点核数；节点门要 `ping` 通且 `evalSupport` 为真 —— 见
`trainer/batch_plan.py:57-61` 与 `trainer/eval_dispatch.py:114`。）

**停止**：■ 中断本 cell → 自动 unregister（`enabled=false` + `unregistered_at`，条目**不删**：下个
会话注册时恢复；要永久停用请在控制台删掉那条）。会话按 `max_session_hours` 到点也会干净收线。


In [ ]:
# @title Battle 采样节点 —— 填 CFG.repo（+ 私有仓 token）后直接 Run
import os, shutil, subprocess, sys, threading, time
from pathlib import Path

CFG = {
    # ── 代码库 ──────────────────────────────────────────────────────────
    # 采样节点要的是 **tools/（TS 导出器）+ src/（wasm 内核）+ nn-training/**：
    # hub 的 `GET /code` 只有 nn-training 下的 .py/.jsonc，而 TS 运行时在 ts_code.zip 里
    # 且只随具体 job 下发 —— 采样节点没有 job，所以直接 clone 全仓是唯一省事的路
    # （云机有外网）。
    "repo": "https://github.com/<owner>/battle2.git",
    "ref": "main",
    "github_token": "",        # 私有仓 PAT；留空则读环境变量 GITHUB_TOKEN
    "code_dir": "/kaggle/working/battle2",   # Colab 建议 /content/battle2
    # ── 网络（GitHub 直连不通时：阿里云 DSW / 国内云机）──────────────────
    # github_base：显式指定前缀式镜像基址（例 "https://gh-proxy.com/https://github.com"）。
    #   留空 = 自动尝试内置镜像（gh-proxy → ghproxy.net），直连不通时无需填任何东西。
    #   私有仓的镜像一般不转发认证 ⇒ 请改用 code_zip。
    "github_base": "",
    # bun_mirror：bun 二进制镜像目录（npmmirror 布局）；留空 = 官方安装失败后自动回落默认镜像。
    "bun_mirror": "",
    # net_proxy：出站代理，例 "http://127.0.0.1:7890"；clone / 公网下载 / 隧道安装共用。
    "net_proxy": "",
    # code_zip：已上传到本机的仓库归档（仓库根打包，zip/tar.gz）；填了跳过全部 git 操作。
    "code_zip": "",
    # ── 网络通道与自动注册 ───────────────────────────────────────────────
    "link": "auto",            # auto | tailscale | cloudflared（auto：Colab 优先 tailscale）
    "node_id": "",             # 留空 = rollout-<env>（colab/kaggle）
    "hub_ip": "",              # 推荐用 secret HUB_IP；"100.x.y.z" 或整条 "http://host:8900"
    "hub_port": 8900,
    "hub_url": "",             # 整条 URL 时填这里（优先于 hub_ip + hub_port）
    "hub_token": "",           # 推荐用 secret HUB_TOKEN（= 训练机 HUB_TOKEN）
    "ts_authkey": "",          # 推荐用 secret TS_AUTHKEY；cloudflared 通道可留空
    "ts_ephemeral": True,      # tailnet 退出即注销（别留僵尸节点名）
    "ts_proxy_env": True,      # userspace 模式注入代理环境（出站可用、入站不通）
    "ts_engine": "",           # 空 = auto（用户态优先）；"kernel" 等可强压
    "ts_no_proxy_extra": "",   # 追加 NO_PROXY（逗号分隔）
    "ts_tarball": "",          # 预置 tarball 路径/URL（内网/离线环境）
    "ts_prefix": "",           # 预置 tailscale 安装前缀
    # ── agent ───────────────────────────────────────────────────────────
    "agent_port": 8443,
    "workers": 0,              # 0 = 自动（platform_utils.cpu_worker_slots；>0 完全照填的数）
    "cache_mb": 2048,          # 结果缓存上限（同键重入直接回放）
    "agent_extra": [],         # 例 ["--no-node"] = 采样子进程强制 bun（A/B 与回滚用）
    "bun_install_deps": False, # agent 零第三方依赖；勾 True 才跑 bun install
    # ── 会话与保活 ───────────────────────────────────────────────────────
    "max_session_hours": 9,    # 云机单次会话上限；到点干净收线（含 unregister）
    "restart_limit": 5,        # agent 崩溃自动重启上限（超过就停 cell，别无限重启）
    "keepalive_every_sec": 120,
    "cloudflared_path": "",    # 留空 = 自动找/自动装（实现住 remote/rollout_node.py）
    "log_file": "",            # 留空 = /content → /kaggle/working → /tmp 自动挑
}

# 日志同时落文件：会话被无声回收时 cell 输出会连着丢（plan §2.5）
def _pick_log_path():
    for _d in ("/content", "/kaggle/working"):
        if Path(_d).is_dir():
            return Path(_d) / "rollout-node.log"
    return Path("/tmp/rollout-node.log")


_LOG_PATH = Path(CFG.get("log_file") or "") if CFG.get("log_file") else _pick_log_path()


def _log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] [rollout-node] {msg}"
    print(line, flush=True)
    try:
        with _LOG_PATH.open("a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except OSError:
        pass


def _secret(key, cfg_val=""):
    """环境变量 → Colab/Kaggle Secrets → CFG 手填（值永不进日志，只记**来源**）。

    ★ 调用发生在 `run()` 的最前面、任何代理注入（tailscale userspace 会改写 HTTP_PROXY）
    之前 —— 顺序是契约（2026-09-17 事故同族），别把凭据读取挪到 clone/引导之后。
    """

    def _env():
        return os.environ.get(key, "")

    def _colab():
        from google.colab import userdata
        return userdata.get(key) or ""

    def _kaggle():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key) or ""

    for _name, _get in (("env", _env), ("colab", _colab), ("kaggle", _kaggle)):
        try:
            _v = _get()
        except Exception as _e:
            _log(f"凭据 {key} ← {_name} 取用失败（{type(_e).__name__}）")
            _v = ""
        if _v:
            _log(f"凭据 {key} ← {_name}")
            return str(_v).strip()
    if cfg_val:
        _log(f"凭据 {key} ← CFG 手填")
    return str(cfg_val or "").strip()


# ── 保活：云机一段时间无输出会被回收 —— 打印本身就是心跳 ────────────────
# （与 battle.cloudflared.ipynb / battle.offline.ipynb 同一招；Colab 那个分支要点连接
#  按钮，Kaggle 只要别静默。）
_stop = threading.Event()


def _keepalive():
    n = 0
    every = max(30, int(CFG["keepalive_every_sec"]))
    while not _stop.is_set():
        _log(f"[keepalive] alive ({n * every // 60} min)")
        n += 1
        _stop.wait(every)


threading.Thread(target=_keepalive, daemon=True, name="keepalive").start()
_log("Keepalive 已启动")


# ── 出站代理（可填 net_proxy）：必须在 clone/任何公网下载之前设好 ──────────
# NO_PROXY 合并平台原值 + 本机回环 + hub 主机（hub 走隧道/内网，不该被塞进公网代理）。
_proxy = str(CFG.get("net_proxy") or "").strip()
if _proxy:
    _no_proxy = ",".join(
        x for x in (
            os.environ.get("NO_PROXY", ""),
            "localhost",
            "127.0.0.1",
            "::1",
            str(CFG.get("hub_ip") or "").strip(),
            str(CFG.get("ts_no_proxy_extra") or "").strip(),
        ) if x
    )
    for _k in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
        os.environ[_k] = _proxy
    os.environ["NO_PROXY"] = os.environ["no_proxy"] = _no_proxy
    _log("出站代理已设置（net_proxy；值不打印）")


# ── 代码库（GITHUB_TOKEN 只在这里用；clone 走平台网络，代理还没注入）────────
def _sh(cmd, *, cwd=None, timeout=None):
    try:
        return subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                            encoding='utf-8', errors='replace', timeout=timeout)
    except (OSError, subprocess.SubprocessError) as e:

        class _R:
            returncode = 127
            stdout = ""
            stderr = str(e)

        return _R()


def _clone_candidates(url):
    """clone 候选（自动，无需配置）：显式 github_base → 直连 → 内置镜像（去重保序）。"""
    if not url.startswith("https://github.com"):
        return [url]
    _rest = url[len("https://github.com"):]
    _base = str(CFG.get("github_base") or os.environ.get("GITHUB_BASE") or "").strip().rstrip("/")
    _out = [_base + _rest] if _base else [url]
    if _base:
        _out.append(url)
    _out += ["https://gh-proxy.com/https://github.com" + _rest,
             "https://ghproxy.net/https://github.com" + _rest]
    return list(dict.fromkeys(_out))


def _probe_clone(item):
    """单候选探活：`(url, 探活 url)` ⇒ 可达返回 `(url, 探活 url, 秒)`。"""
    _u, _probe = item
    _t0 = time.monotonic()
    _rc = _sh(
        ["git", "ls-remote", "--exit-code", _probe, "HEAD"], timeout=25
    ).returncode
    return (_u, _probe, time.monotonic() - _t0) if _rc == 0 else None


def _rank_clone_candidates(cands, probe):
    """并行探活（不串行白等墙掉的链路），按耗时升序返回可达候选。"""
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=max(1, len(cands))) as _ex:
        _hits = [h for h in _ex.map(probe, cands) if h]
    _hits.sort(key=lambda h: h[2])
    return _hits


code_dir = Path(CFG["code_dir"])
repo = str(CFG["repo"] or "").strip()
ref = str(CFG["ref"] or "main").strip() or "main"
_zip_src = str(CFG.get("code_zip") or "").strip()
code_dir.parent.mkdir(parents=True, exist_ok=True)
if _zip_src:
    # 完全离线路径：仓库归档已上传（仓库根打包）。跳过 clone，连 .git 都不需要。
    _zip = Path(_zip_src).expanduser()
    if not _zip.is_file():
        raise SystemExit(f"[rollout-node] FATAL: code_zip 不存在: {_zip}")
    if (code_dir / "tools" / "agent").is_dir():
        _log(f"code_zip 已解压，复用: {code_dir}")
    else:
        import tarfile, zipfile
        code_dir.mkdir(parents=True, exist_ok=True)
        _log(f"解压代码库归档: {_zip} → {code_dir}")
        if zipfile.is_zipfile(_zip):
            with zipfile.ZipFile(_zip) as _z:
                _z.extractall(code_dir)
        else:
            with tarfile.open(_zip, "r:*") as _t:
                try:
                    _t.extractall(code_dir, filter="data")
                except TypeError:   # Python < 3.12 没有 filter 参数
                    _t.extractall(code_dir)
    _head = ""
elif (code_dir / ".git").is_dir():
    _sh(["git", "-C", str(code_dir), "fetch", "--depth", "1", "origin", ref], timeout=600)
    _sh(["git", "-C", str(code_dir), "checkout", "--force", ref], timeout=300)
    _sh(["git", "-C", str(code_dir), "reset", "--hard", f"origin/{ref}"], timeout=300)
    _head = _sh(["git", "-C", str(code_dir), "rev-parse", "--short", "HEAD"]).stdout.strip()
else:
    if (not repo) or "<owner>" in repo:
        raise SystemExit("[rollout-node] FATAL: CFG.repo 必填（形如 https://github.com/<owner>/battle2.git）")
    _tok = str(CFG["github_token"] or "").strip() or os.environ.get("GITHUB_TOKEN", "")
    _cands = []
    for _u in _clone_candidates(repo):
        _probe = _u
        if _tok and _u == repo and _u.startswith("https://"):
            _probe = repo.replace("https://", f"https://{_tok}@")   # 私有仓；只在内存里拼，不落日志
        _cands.append((_u, _probe))
    _hits = _rank_clone_candidates(_cands, _probe_clone)
    if not _hits:
        _log("git clone 失败：直连与内置镜像都不通。")
        _log("  ① 企业代理/出网策略 ⇒ 填 CFG.net_proxy 后重跑")
        _log("  ② 私有仓/完全离线 ⇒ 上传仓库归档后填 CFG.code_zip")
        raise SystemExit(2)
    _had_dir = code_dir.exists()
    for _pick, _pick_url, _secs in _hits:
        _log(f"clone 源: {_pick}（并行探活 {_secs:.1f}s）")
        if _tok and _pick != repo:
            _log("!! 镜像一般不转发认证 —— 私有仓请改用 code_zip")
        r = _sh(["git", "clone", "--depth", "1", "--branch", ref, _pick_url, str(code_dir)], timeout=900)
        if r.returncode == 0:
            break
        _log(f"clone 失败，换下一个源: {_pick}")
        _log(r.stderr[-600:])
        if not _had_dir:
            shutil.rmtree(code_dir, ignore_errors=True)
    else:
        _log("git clone 失败：所有可达源都没克隆成功。")
        raise SystemExit(2)
    _head = _sh(["git", "-C", str(code_dir), "rev-parse", "--short", "HEAD"]).stdout.strip()
_log(f"代码库就绪: {code_dir}  commit={_head or '(无 git)'}  ref={ref}")


# ── 调用仓库里的运行时：bun / cloudflared / 通道 / agent / 注册 / 监护全在那边 ──
sys.path.insert(0, str(code_dir / "nn-training"))
from remote import rollout_node  # noqa: E402  （必须在 clone 之后）

try:
    _rc = rollout_node.run(CFG, _log, _secret)
finally:
    _stop.set()   # 停心跳（进程退出时 finally 里的 unregister/清理由 run 负责）
_log(f"退出码 {_rc}（0 = 正常收线）")
